# 02. Startup Scenario and Vacuum Fields

A tokamak discharge does not simply happen. Coils are charged, gas is admitted, a loop
voltage is induced, and if the field at the moment of breakdown has the right shape, a
plasma forms and carries current. This session reads one VEST discharge backwards: from the
signals that show what the plasma did, to the fields that let it start at all.


## Session Overview

By the end of this session you will be able to:

- read the plasma-response signals of a discharge and say when the plasma formed;
- separate the current in the machine into what the coils drove, what the vessel carried,
  and what the plasma did;
- compute the vacuum quantities that decide whether a startup can work at all — the loop
  voltage, the vertical field, and the field decay index;
- check that the vacuum-field model reproduces the magnetics actually measured, before
  trusting anything derived from it;
- find the breakdown time from the data rather than by eye.

Everything runs offline on the packaged VEST discharge 39915.


## Physical Context

### Getting a plasma started

Startup has three phases, and each leaves a different fingerprint in the diagnostics.

**Breakdown.** The central solenoid swings, inducing a toroidal electric field. Free
electrons accelerate, ionise the fill gas, and an avalanche runs away. Whether it does turns
on three numbers together — the field, the fill pressure, and how far a line runs before it
meets a wall — and the last of them is why the field null matters. The session works all
three out at the end.

**Burn-through.** The young plasma is cold and full of neutrals and impurities, which radiate
away much of the ohmic input. It either heats through this barrier or it dies. Line
radiation, which the filterscope sees, is the direct evidence.

**Current ramp-up.** Past burn-through the plasma is conducting well, and the transformer
drives its current up.

### The vacuum field, and why it is a *precondition*

Before any plasma exists, the coils and the vessel already make a field. Three properties of
it decide whether startup is possible: the **loop voltage**, which supplies the drive; the
**vertical field** $B_z$, which balances the plasma's outward hoop force once current flows;
and the **decay index**, which decides whether that balance is stable rather than merely
present. Each is written down where the session computes it, beside the figure it explains.

### The vessel is part of the circuit

VEST's vacuum vessel is a conductor. A changing coil current induces eddy currents in it,
and those currents make a field of their own. No honest vacuum-field calculation can leave
them out — and nothing in the file records them, so the session has to solve for them before
it can read the vacuum field at all.


## Load / Prepare Data

### The discharge


In [ ]:
import numpy as np
import vaft
import matplotlib.pyplot as plt


In [ ]:
ods = vaft.omas.sample_ods()
sorted(ods.keys())


## Guided Analysis

### What the plasma did

Start with the response signals. The plasma current says whether there was a plasma; the
line emission says what state it was in.


In [ ]:
vaft.omas.plot_plasma_current_time(ods)
plt.show()


In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha")
plt.show()


H-alpha is neutral hydrogen at the edge — recycling and fuelling. Impurity lines tell the
burn-through story instead: carbon and oxygen come off the wall, radiate, and have to be
overcome.


In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="CIII")
plt.show()


### What drove it

Two coil systems. The toroidal field coil sets the field the plasma sits in; the poloidal
field coils drive and shape it.


In [ ]:
vaft.omas.plot_tf_coil_time_current(ods)
plt.show()


In [ ]:
vaft.omas.plot_pf_coil_time_current(ods)
plt.show()


Two other actuators belong in this picture and are not here: the **gas valve command** and
the **EC pre-ionisation trigger**. Both are recorded on VEST but neither is mapped into IMAS
yet, so there is nothing to plot. Their absence is a mapping gap, not a quiet discharge.

### Where the current actually is

At any moment the machine carries current in three places, and only one of them is the
plasma. Separating them is the first real analysis step of the session.


### Solving the vessel currents

You have now seen two of the three currents the machine carries: the coils, which were
commanded, and the plasma, which answered. The third is in neither trace. The packaged shot
carries 950 passive loops describing the vessel and no currents in them — solving that
circuit is a modelling step rather than a measurement, so nothing stores the result.

Passing empty plasma filaments computes the **vacuum** case: the vessel's response to the
coils alone, with no plasma. That is exactly the field a startup has to work with.

What the solve integrates is a set of coupled circuits,

$$\mathbf{M}\,\frac{\mathrm{d}\mathbf{I}_v}{\mathrm{d}t}
  = -\,\mathbf{R}\,\mathbf{I}_v - \mathbf{L}\,\frac{\mathrm{d}\mathbf{I}_c}{\mathrm{d}t},$$

with $\mathbf{M}$ the passive-passive mutual inductance matrix, $\mathbf{L}$ the
passive-active one, and $\mathbf{R}$ the diagonal loop resistance. Two things follow from the
shape of it. The drive is $\mathrm{d}\mathbf{I}_c/\mathrm{d}t$ and not $\mathbf{I}_c$, so a
steady coil current induces nothing at all. And the homogeneous response decays on the
eigenvalues of $\mathbf{M}\mathbf{R}^{-1}$ — the wall's own $L/R$ times — which is why the
vessel field lags the coils rather than following them.


In [ ]:
before = {row["name"] for row in vaft.omas.available_plots(ods)}
print("passive current available?", "passive_structure_time_current" in before)
print("pf_passive loops        :", len(ods["pf_passive.loop"]))

vaft.omas.compute_eddy_currents(ods, [], [])

unlocked = {row["name"] for row in vaft.omas.available_plots(ods)} - before
print("time points solved:", len(ods["pf_passive.time"]))
print("plots unlocked    :", sorted(unlocked))

Three views appeared, and the reason is worth noticing: none of them is about the vessel
itself. Two are magnetics comparisons and one is the vessel current. They became available
because the vacuum-field model they rest on now exists — and the next two figures are two of
them.


In [ ]:
vaft.omas.plot_current_overview(ods)
plt.show()


The vessel current is large — comparable to the plasma current — and it is *induced*, so it
opposes whatever the coils just did. A magnetic measurement made outside the vessel sees the
sum of all three. That is the whole difficulty of magnetic reconstruction in one figure.

Vacuum fields superpose, which is the only reason that sum can be taken apart again:

$$\psi(r, z) = \psi_{\text{coils}} + \psi_{\text{vessel}} + \psi_{\text{plasma}},
  \qquad \psi_i = \mu_0\,G(r, z; r_i, z_i)\,I_i,$$

where $G$ is the Green's function of a circular filament. Each $\psi_i$ is a *response per
unit current* — geometry only, carrying no current of its own. The first two terms are
computable once the currents are known; the third is what a reconstruction is after, and it
is only ever reached by subtracting the other two.

### Does the vacuum model match the machine?

Everything below this point is derived from the field model. Before trusting it, check it
against the magnetics that were actually measured. The next figure puts three curves on each
channel:

- **measured** — what that probe or flux loop actually recorded;
- **coil-only** — the synthetic signal from `pf_active` alone;
- **coil + eddy** — the same with the vessel currents solved for above added in.

The gap between the second and the third *is* the vessel, drawn directly. It is also the
error a reconstruction inherits if the vessel is left out, and because it is induced it does
not look like noise: it looks like current, in the place a plasma would be. Where the third
curve reaches the first, the vacuum model is complete and what remains is plasma. Where it
does not, the leftover is either plasma or a defect in the model, and this figure alone
cannot say which.


In [ ]:
vaft.omas.plot_magnetics_overview_vacuum(ods)
plt.show()


### The drive

The loop voltage is the time derivative of poloidal flux at a point — literally what the
transformer is doing to the plasma's future location. The field it corresponds to is

$$E_\varphi = -\frac{1}{2\pi R}\frac{\partial\psi}{\partial t},
  \qquad V_{\mathrm{loop}} = \oint \mathbf{E}\cdot\mathrm{d}\boldsymbol{\ell}
  = 2\pi R\,E_\varphi = -\frac{\partial\psi}{\partial t},$$

with $\psi$ in weber. Whether that $2\pi$ survives depends on whether the flux is stored per
weber or per radian, and VAFT's two kernels disagree on it *and* on the sign:
`toroidal_electric_field` is the expression above, while `loop_voltage_from_total_flux` is
$+2\pi\,\mathrm{d}\psi_b/\mathrm{d}t$ on a per-radian flux. That is
[#354](https://github.com/VEST-Tokamak/vaft/issues/354), still open. The cell below takes the
first.


In [ ]:
time, v_loop = vaft.omas.compute_startup_loop_voltage_ods(ods, rz=(0.4, 0.0))

figure, axes = plt.subplots()
axes.plot(time, v_loop)
axes.set_xlabel("Time [s]")
axes.set_ylabel(r"$V_{\mathrm{loop}}$ [V]")
axes.grid(alpha=0.3)
plt.show()

print(f"peak |V_loop| = {np.nanmax(np.abs(v_loop)):.2f} V "
      f"at t = {time[np.nanargmax(np.abs(v_loop))]:.4f} s")


### The vertical field and its decay index

A plasma ring wants to expand. The vertical field pushes back, and whether that restoring
force is *stable* depends on how fast the field falls off with radius. Both quantities come
out of the same flux map:

$$B_Z = -\frac{k}{R}\frac{\partial\psi}{\partial R},
  \qquad n = -\frac{R}{B_Z}\frac{\partial B_Z}{\partial R},
  \qquad k = \frac{\sigma_{R\varphi Z}\,\sigma_{B_p}}{(2\pi)^{e_{B_p}}},$$

where $k$ carries the COCOS orientation and the $2\pi$ together. So $n$ is a second
derivative of $\psi$ wearing a disguise — which is why it is the noisiest quantity in this
session, and why it has no value at all on the surface where $B_Z$ changes sign.

The window $0 < n < 1.5$ is the rigid-ring result: below zero the vertical field does not
restore a radial displacement at all, above $1.5$ the ring is unstable to vertical motion. It
assumes a thin current ring and no conducting wall. VEST has a very conducting wall — the
same one solved for above — so the true window is wider than the band drawn below.


In [ ]:
radius, decay_index = vaft.omas.compute_decay_index_ods(ods, time=0.3307)

figure, axes = plt.subplots()
axes.plot(radius, decay_index)
axes.axhspan(0.0, 1.5, alpha=0.15, label="passively stable")
axes.set_xlabel("R [m]")
axes.set_ylabel("decay index $n$")
axes.legend()
axes.grid(alpha=0.3)
plt.show()

finite = np.isfinite(decay_index)
print(f"n spans {decay_index[finite].min():.2f} to {decay_index[finite].max():.2f}")
print("entirely inside 0 < n < 1.5:",
      bool(np.all((decay_index[finite] > 0.0) & (decay_index[finite] < 1.5))))


At this instant the index is inside the stable band across the whole radial range, so a
current ring here would be held rather than flung out. Note which instant that is: 330.7 ms,
some 24 ms after the plasma formed. At the breakdown onset the same cut leaves the band — the
vertical field is only a few gauss there and falls off steeply — but at breakdown there is no
current ring to hold yet, so the question the decay index answers has not started. The vacuum
map further down shows the whole sweep. Note also that $n$ is undefined where $B_z$ crosses
zero — the calculation returns `nan` there rather than a large meaningless number.

### When did the plasma form?

Reading a breakdown time off an H-alpha trace by eye is a habit worth breaking. The onset can
be found from the data.


In [ ]:
t_breakdown = vaft.omas.find_breakdown_onset(ods)
print(f"breakdown onset: {t_breakdown * 1e3:.2f} ms")


That one number is not a threshold read off one trace. `find_breakdown_onset` is a thin wrapper
over `plasma_timing`, and the object it throws away is where the criterion lives:

- **H-alpha, chosen by label, is authoritative.** The detector asks for the line called
  `H-alpha_6563`, never for a channel number, and prefers the slowest digitizer that carries it.
  Light is optical, so the coil-firing pickup every magnetic diagnostic carries cannot trigger it.
- **The plasma current is the fallback, and always the cross-check.** It is computed whether or
  not the light answered, and the verdict records whether the two agree.
- **A raw magnetic first crossing is never used.** It fires on the coils, not on the plasma.

The test is not a derivative. A sample has to exceed
$\text{baseline} + \max(f\cdot\text{peak},\ \sigma\cdot\sigma_{\text{robust}})$, *stay* above it for
half a millisecond, and belong to a run wide, prominent and large enough to be a pulse rather
than a spike. The fraction-of-peak term is what makes two channels with a ten-fold difference in
noise agree on the onset.

In [ ]:
from vaft.omas.plasma_timing import plasma_timing
from vaft.omas.discharge_timing import discharge_timing

timing = plasma_timing(ods)
events = discharge_timing(ods)

print(f"ohmic coil fired   : {events.oh_onset * 1e3:7.2f} ms  ({events.oh_coil})")
print(f"loop voltage zero  : {events.vloop_time * 1e3:7.2f} ms")
print(f"plasma window      : {timing.onset * 1e3:7.2f} - {timing.offset * 1e3:.2f} ms")
print(f"decided by         : {timing.source}")
print(f"light and current  : {timing.agreement}; the current started "
      f"{abs(timing.onset_delta_s) * 1e6:.0f} us "
      f"{'before' if timing.onset_delta_s < 0 else 'after'} the light")
print(f"above threshold    : {timing.duty_cycle:.0%} of the window")

Read it as a sequence. The ohmic coil fires; some ten milliseconds later the loop voltage
crosses zero; the gas breaks down about four milliseconds after that. The light and the current
agree to within a single sample — `consistent` — and the window is above threshold for all
of its length, so its duration really is a duration and not an envelope drawn around gaps
([#752](https://github.com/VEST-Tokamak/vaft/issues/752)).

The residual figure below marks an "Ip onset" of its own, and it is a **different detector**: a
five-sigma first crossing of each signal's pre-plasma noise band, applied identically to every
residual so that they can be compared with one another. On another shot the two numbers need not
agree. On this one they coincide.

In [ ]:
vaft.omas.plot_magnetics_overview_plasma_residual(ods)
plt.show()


### The field null

Breakdown needs somewhere for electrons to accelerate without promptly hitting a wall — a
region of weak poloidal field with long connection lengths. In the vacuum flux map it shows
up as the null: the other field component comes from the same map with the other derivative,
and the null is where the pair of them vanishes together,

$$B_R = \frac{k}{R}\frac{\partial\psi}{\partial Z},
  \qquad |B_p| = \sqrt{B_R^2 + B_Z^2} \longrightarrow 0 .$$

It is not a point of zero *total* field. $B_\varphi$ is untouched by everything above and is
two orders of magnitude larger there. A null is where the field becomes almost purely
toroidal, so a line wraps the torus many times before walking out to a wall — which is what
buys the connection length the next section turns out to depend on entirely.


In [ ]:
vaft.omas.plot_equilibrium_field_psi_vacuum(ods, time=t_breakdown)
plt.show()


### The vacuum field in time

A single frame hides the most important fact about the null: it does not last. The canonical map,
`plot_vacuum_field`, draws six quantities of the coils' and the vessel's field from one
evaluation — the flux, $|B_p|$, the decay index, $|E_\varphi|$, a breakdown figure of merit and
the Lloyd margin — and caches the grid's response to the coils, so moving in time costs a
contraction rather than a solve. That is what makes it cheap enough to sweep.

Start with the single number that says how good a null is: how much of the vessel has a poloidal
field weaker than five gauss.

In [ ]:
from matplotlib.path import Path as MplPath

limiter = MplPath(np.column_stack([
    ods["wall.description_2d.0.limiter.unit.0.outline.r"],
    ods["wall.description_2d.0.limiter.unit.0.outline.z"],
]))
pf_time = np.asarray(ods["pf_active.time"])
around = int(np.argmin(np.abs(pf_time - t_breakdown)))
sweep_index = np.arange(around - 60, around + 61, 3)

weak_fraction = []
for index in sweep_index:
    field = vaft.omas.compute_vacuum_field_map(ods, time=float(pf_time[index]), resolution=33)
    mesh_r, mesh_z = np.meshgrid(field["r"], field["z"], indexing="ij")
    inside = limiter.contains_points(
        np.column_stack([mesh_r.ravel(), mesh_z.ravel()])).reshape(mesh_r.shape)
    b_p = vaft.formula.poloidal_field_magnitude(field["b_r"], field["b_z"])
    weak_fraction.append(np.mean(b_p[inside] < 5e-4))
weak_fraction = np.asarray(weak_fraction)
sweep_time = pf_time[sweep_index]
t_null = float(sweep_time[np.argmax(weak_fraction)])

figure, axes = plt.subplots()
axes.plot(sweep_time * 1e3, weak_fraction * 100)
axes.axvline(t_breakdown * 1e3, color="k", ls="--", lw=0.8, label="onset, from the light")
axes.set_xlabel("Time [ms]")
axes.set_ylabel(r"vessel with $|B_p| < 5$ G [%]")
axes.legend()
axes.grid(alpha=0.3)
plt.show()

print(f"widest null   : {t_null * 1e3:.2f} ms, {weak_fraction.max():.0%} of the vessel below 5 G")
print(f"light appears : {(t_breakdown - t_null) * 1e3:+.2f} ms after it")

The null is not an object the field has; it is a moment the field passes through. For about a
millisecond a large part of the vessel has almost no poloidal field, and then it closes again.
The widest point comes a fraction of a millisecond *before* the light — which is the right order.
The field opens, the gas inside the opening avalanches, and the H-alpha detector sees the result.
Nothing forced the two curves to line up: the onset was timed from the light alone, and this
curve comes from the coils alone.

Here are four of the quantities at that instant.

In [ ]:
panels = {"psi": r"$\psi$", "b_poloidal": r"$|B_p|$",
          "decay_index": "decay index $n$", "e_toroidal": r"$|E_\varphi|$"}
figure, axes = plt.subplots(1, 4, figsize=(16, 6))
for axis, (quantity, name) in zip(axes, panels.items()):
    vaft.omas.plot_vacuum_field(ods, field=quantity, time=t_null, resolution=65, ax=axis)
    axis.set_title(name)
figure.suptitle(f"The vacuum field at the widest null, t = {t_null * 1e3:.1f} ms")
plt.show()

In [ ]:
# interaction_backend="auto" gives a live time slider in Jupyter. This cell drives the
# same control from code, so it also runs headless and prints what the slider shows.
result = vaft.omas.plot_vacuum_field(
    ods, field="b_poloidal", resolution=33,
    interactive=True, interaction_backend="none",
)
slider = next(control for control in result.controls if control.name == "time_index")
print(f"{slider.label}: {slider.options[1] + 1} positions")
for index in (around - 30, around, around + 30):
    result.state.set("time_index", int(index))
    print(f"  {result.axes.get_title()}")
plt.close(result.figure)

### Was that drive enough?

"Weak poloidal field and long connection lengths" is a description, not a test. Whether an
electron avalanches before it reaches a wall turns on three numbers: how hard it is pushed
($E_\parallel$), how much gas there is to ionise ($p$), and how far it travels before
hitting something ($L$). Two of the three are in the file — the pressure has been there
since the first cell listed `barometry`, and the session has not looked at it once. The third
has to be traced.

In [ ]:
pressure = ods["barometry.gauge.0.pressure.data"]
pressure_time = ods["barometry.gauge.0.pressure.time"]

figure, axes = plt.subplots()
axes.plot(pressure_time, pressure * 1e3)
axes.axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes.set_xlabel("Time [s]")
axes.set_ylabel("Pressure [mPa]")
axes.grid(alpha=0.3)
plt.show()

prefill = float(np.median(pressure[pressure_time < 0.29]))
at_onset = float(np.median(pressure[(pressure_time > 0.300) & (pressure_time < 0.312)]))
molecules = vaft.formula.neutral_density_from_pressure(prefill)

print(f"prefill (median before 0.29 s): {prefill:.3e} Pa "
      f"= {prefill / vaft.formula.PA_PER_TORR:.2e} Torr")
print(f"H2 molecules: {molecules:.2e} m^-3, "
      f"hydrogen atoms: {vaft.formula.atomic_inventory_from_molecular_gas(molecules):.2e} m^-3")
print(f"the same gauge at the breakdown instant: {at_onset:.3e} Pa "
      f"({at_onset / prefill:.2f}x the prefill)")

The gauge never settles. It rises through the whole record and peaks *after* breakdown, as
the discharge drives gas off the wall faster than a Penning gauge can follow, so "the
prefill" is a choice of window rather than a reading — worth remembering when the numbers
below start looking precise.

The third quantity is not in the file. Before tracing it, look at the estimate everyone
reaches for first, because it is worth knowing how far it can be trusted. A line at pitch
$B_\perp / B_T$ needs $B_T / B_\perp$ turns to drift across a region of size $a$, which gives

$$L_{\mathrm{open}} \sim a\,\frac{B_T}{B_\perp}.$$

No wall in it, no null geometry, no constant of order one. It is worth plotting for its
*slope* — what a better null buys — and the trace below says how far to trust it as a number.

In [ ]:
r_probe = 0.4  # the radius the loop voltage above was evaluated at
b_toroidal = float(ods["tf.b_field_tor_vacuum_r.data"][
    int(np.argmin(np.abs(ods["tf.time"] - 0.3307)))]) / r_probe
limiter_r = ods["wall.description_2d.0.limiter.unit.0.outline.r"]
minor_radius = 0.5 * float(limiter_r.max() - limiter_r.min())

b_perp = np.geomspace(1e-4, 5e-3, 200)  # 1 G to 50 G at the null
length_open = minor_radius * b_toroidal / b_perp
e_threshold = vaft.formula.lloyd_breakdown_field(prefill, length_open)
e_measured = float(np.nanmax(np.abs(v_loop))) / (2 * np.pi * 0.4)

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(b_perp * 1e4, length_open)
axes[0].set_ylabel(r"$L_{\mathrm{open}}$ [m]")
axes[1].loglog(b_perp * 1e4, e_threshold)
axes[1].axhline(e_measured, color="k", ls="--", lw=0.8)
axes[1].set_ylabel(r"$E_{\mathrm{BD}}$ [V/m]")
axes[1].set_xlim(axes[0].get_xlim())  # so the right curve visibly stops early
for axis in axes:
    axis.set_xlabel(r"$B_\perp$ at the null [G]")
    axis.grid(alpha=0.3, which="both")
plt.show()

print(f"B_T at R = 0.4 m: {b_toroidal:.4f} T")
print(f"measured drive at the same radius: E = {e_measured:.3f} V/m")

It no longer has to be assumed. `trace_field_line` follows a line through a field and stops it
at the wall; what it could not do before breakdown was get a field to follow, because the only
field it knew how to build came from an equilibrium, and a discharge that has not broken down has
none. `compute_connection_length_map_ods` builds the vacuum field instead and traces a line from
every point of a grid, in both directions, to the limiter. It takes a few seconds, which is why it
is a one-off here and not a slider frame.

In [ ]:
traced = vaft.omas.compute_connection_length_map_ods(ods, time=t_breakdown, resolution=33)
interior = ~traced["outside"]
open_lines = interior & ~traced["saturated"]

field = vaft.omas.compute_vacuum_field_map(ods, time=t_breakdown, resolution=33)
b_p = vaft.formula.poloidal_field_magnitude(field["b_r"], field["b_z"])
scaling = minor_radius * (b_toroidal * r_probe / field["r"][:, None]) / b_p
ratio = scaling[open_lines] / traced["length_m"][open_lines]

figure, axes = plt.subplots()
axes.loglog(traced["length_m"][open_lines], scaling[open_lines], ".", ms=3, alpha=0.5)
axes.plot([1.0, 300.0], [1.0, 300.0], "k--", lw=0.8, label="scaling = trace")
axes.set_xlabel("traced connection length [m]")
axes.set_ylabel(r"$a B_T / B_\perp$ scaling [m]")
axes.legend()
axes.grid(alpha=0.3, which="both")
plt.show()

print(f"traced from {interior.sum()} points; {traced['saturated'].sum()} never met the wall")
print(f"scaling / trace: median {np.median(ratio):.2f}, "
      f"half of them between {np.percentile(ratio, 25):.2f} and {np.percentile(ratio, 75):.2f}")
print(f"log-log correlation: {np.corrcoef(np.log(scaling[open_lines]), np.log(traced['length_m'][open_lines]))[0, 1]:.2f}")

Both curves stop, and that is not a plotting failure. Below $A p L = 1$ the avalanche
cannot close *at any field*, so `lloyd_breakdown_field` warns and returns `nan` instead of
a large number — the model has a domain, and its edge is a physical statement. At this
prefill, a connection length shorter than about a hundred metres means no breakdown however
hard you push.

Turn the picture around and hold $L$ fixed instead. The threshold is then a curve in
pressure with a minimum: too little gas and there is nothing to ionise, too much and the
electrons collide before they have gained the ionisation energy. It is a Paschen curve with
the electrode gap replaced by the connection length.

In [ ]:
p_scan = np.geomspace(1e-3, 3e-2, 300)

figure, axes = plt.subplots()
for length in (100.0, 200.0, 400.0, 1000.0):
    axes.loglog(p_scan * 1e3, vaft.formula.lloyd_breakdown_field(p_scan, length),
                label=f"L = {length:.0f} m")
axes.axhline(e_measured, color="k", ls="--", lw=0.8)
axes.axvline(prefill * 1e3, color="k", ls=":", lw=0.8)
axes.set_xlabel("Prefill pressure [mPa]")
axes.set_ylabel(r"$E_{\mathrm{BD}}$ [V/m]")
axes.legend()
axes.grid(alpha=0.3, which="both")
plt.show()

length_scan = np.linspace(80.0, 400.0, 32001)
threshold = vaft.formula.lloyd_breakdown_field(prefill, length_scan)
margin = vaft.formula.breakdown_margin(e_measured, threshold)
needed = length_scan[margin >= 1.0].min()

print(f"no threshold exists at all below L = {length_scan[np.isfinite(threshold)].min():.2f} m")
print(f"the measured drive is enough from L = {needed:.2f} m, "
      f"i.e. a null better than {minor_radius * b_toroidal / needed * 1e4:.2f} G")
print(f"margin at L = 100 m: {np.interp(100.0, length_scan, margin):.2f}, "
      f"at L = 120 m: {np.interp(120.0, length_scan, margin):.2f}")
print(f"doubling the drive would only move the requirement to "
      f"{length_scan[vaft.formula.breakdown_margin(2 * e_measured, threshold) >= 1.0].min():.2f} m")
print(f"the fit is being read at E/p = "
      f"{e_measured / (prefill / vaft.formula.PA_PER_TORR) / 100:.0f} V/cm/Torr")

In [ ]:
figure, axes = plt.subplots(figsize=(5, 6))
vaft.omas.plot_vacuum_field(ods, field="lloyd_margin", time=t_breakdown, resolution=33, ax=axes)
plt.show()

e_local = np.abs(vaft.formula.toroidal_electric_field(field["r"][:, None], field["dpsi_dt"]))
threshold = vaft.formula.lloyd_breakdown_field(prefill, traced["length_m"])
margin = vaft.formula.breakdown_margin(e_local, threshold)
has_threshold = interior & np.isfinite(margin)
print(f"a threshold exists over {has_threshold.sum() / interior.sum():.0%} of the vessel")
print(f"where one exists, the drive clears it at {np.mean(margin[has_threshold] >= 1):.0%} of the points")

Read those numbers together and the connection length turns out to be the whole answer. At
this prefill and this drive, breakdown becomes possible somewhere around 112 m of
connection length, which the scaling above translates into a null better than roughly four
gauss. Every one of those figures is soft: the gauge rises by nearly a fifth between the
window used here and the breakdown instant, and carrying that higher pressure through moves
the requirement to about 97 m.

One caveat is not about the data at all. Lloyd's constants are a two-parameter fit
to hydrogen over a band of $E/p$, and the printed $E/p$ above sits at the top of where such
fits are usually quoted — so the threshold here is an extrapolation, and it gets worse, not
better, as the curve approaches its domain edge. That is a reason to trust the *ordering* of
operating points and not the third significant figure.

What does not move is *which* quantity decides, and the margin map above shows it directly.
Where a threshold exists at all the drive clears it almost everywhere; what decides breakdown is
whether a line runs far enough for a threshold to exist, and that is the blank part of the map —
part of the answer, not missing data. Doubling the loop voltage would buy seven metres, because
the drive only enters through a logarithm; halving $B_\perp$ doubles the connection length
outright.

### What the camera saw

Everything so far has been read off coils, probes and a gauge. The fast camera recorded the
discharge directly, and 66 of its frames from this shot ship with the VAFT repository — not with
the installed package, so this part needs a clone. They are not in `sample_ods()` either; the
first cell below writes a handful of them into the `camera_visible` IDS, which is the form every
camera plot reads.

In [ ]:
import cv2
from vaft.machine_mapping.camera_visible import (
    vfit_camera_visible_dynamic,
    vfit_camera_visible_static,
)

# Every eleventh packaged frame: six across the discharge. OMAS stores a frame as int64,
# about 10 MB each, so all 66 would be most of a gigabyte for a slider that needs a few.
frames = vaft.data.sample_camera_visible_frame_paths(39915)[::11]
images = [cv2.imread(str(path), cv2.IMREAD_GRAYSCALE) for _, path in frames]
frame_times = [time_s for time_s, _ in frames]

vfit_camera_visible_static(
    ods, lines_n=images[0].shape[0], columns_n=images[0].shape[1], channel_name="Fast Camera",
)
vfit_camera_visible_dynamic(ods, images=images, times_s=frame_times)
print(f"{len(images)} frames: {', '.join(f'{t * 1e3:.1f}' for t in frame_times)} ms")

In [ ]:
overlay_index = int(np.argmin(np.abs(np.asarray(frame_times) - 0.320)))
vaft.omas.plot_camera_visible_image_efit_overlay(ods, shot=39915, frame_index=overlay_index)
plt.show()
print(f"frame at {frame_times[overlay_index] * 1e3:.1f} ms, with the wall, the last closed flux "
      f"surface and the magnetic axis projected through the calibrated pose")

In [ ]:
# interaction_backend="auto" gives a live frame slider in Jupyter: drag it and the discharge
# plays back. This cell steps the same control from code, so it also runs headless.
result = vaft.omas.plot_camera_visible_image(ods, interactive=True, interaction_backend="none")
slider = next(control for control in result.controls if control.name == "frame_index")
print(f"{slider.label}: {slider.options[1] + 1} frames")
for index in range(len(images)):
    result.state.set("frame_index", index)
    print(f"  {result.axes.get_title()}")
plt.close(result.figure)

The first packaged frame is 305.2 ms, just over a millisecond before the onset the light gave,
and it is washed out; from the next one on the vessel is lit from inside. Two things in these frames are not
plasma and are easy to read as if they were: the bright vertical band is the centre stack, and the
reconstruction overlaid above only exists from 316 ms, when the packaged equilibrium starts —
nothing here reconstructs the first ten milliseconds of the discharge.

## Interpretation Checkpoints

Each of these has a definite answer in what you have already plotted.

1. **Compare the breakdown time with the H-alpha rise.** Do they agree? Which would you
   trust to define "the plasma started", and why?
2. **The vessel current opposes the coil current.** Look at the three-way decomposition:
   at what point in the discharge is the vessel contribution largest, and what is the
   transformer doing then?
3. **The loop voltage peaks before the plasma current does.** Should it? What does the delay
   between them tell you about the plasma's inductance?
4. **The decay index sits inside the stable band everywhere.** What would the figure look
   like for a machine where it did not, and what would happen to the plasma?
5. **`compute_eddy_currents` unlocked the magnetics comparison plots.** Why should a
   *modelling* step change what you are allowed to plot?


## Integrated Analysis

### Was this a good startup?

You now have every piece needed to answer that as a physicist rather than by impression.
Assemble them on one time axis and read the sequence.


In [ ]:
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 8))

vaft.omas.plot_plasma_current_time(ods, ax=axes[0])
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha", ax=axes[1])
axes[2].plot(time, v_loop)
axes[2].set_ylabel(r"$V_{\mathrm{loop}}$ [V]")
axes[2].grid(alpha=0.3)

for axis in axes:
    axis.axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes[2].set_xlabel("Time [s]")
plt.show()


The dashed line is the breakdown time found from the data. Read left to right: the loop
voltage is applied first, the plasma forms, H-alpha rises as the young plasma recycles
against the wall, and the current ramps.

The story this discharge tells is a startup that worked, in a vacuum field that was stable
everywhere it could have formed.


## Independent Exercise

### Reconstruct another shot's operating conditions

Take a different VEST discharge and repeat this workflow: solve the vessel currents, check
the vacuum model against the magnetics, find the breakdown time, and decide from the decay
index whether the field it formed in was stable. Then say, in a sentence, what kind of
startup it was.

Loading another shot needs database access, so the cells below are left for you to run in
lab mode.

### What #230 asked for, and what is left

Issue [#230](https://github.com/VEST-Tokamak/vaft/issues/230) asked for two more quantities.
This session now computes one of them; the other is still better named than approximated.

**Connection length.** How far a field line travels before striking a surface. It is traced
above, from every point of the vessel, through the vacuum field at the breakdown instant. What it
does not do is tell a line that never meets the wall apart from one that is merely long: both
stop at the tracing limit and are flagged `saturated`. For breakdown the difference does not
matter — the Lloyd threshold has long since stopped changing there — but anything that needs the
true length of a long line would have to tell them apart.

**The 2.45 GHz EC resonance layer.** Where the electron cyclotron frequency matches the
pre-ionisation source, given the toroidal field. This is a one-line calculation from
$B_t(R)$ — but VEST's diagnostic registry describes no 2.45 GHz system at all, so there is
nothing to place the layer *for*. That looks like a mapping gap rather than a physics one,
and is worth confirming before anyone implements it.


In [ ]:
# 1. Load another discharge. Needs HSDS access, so this is lab mode.
# other = vaft.database.load(41672)

# 2. Solve its vessel currents in the vacuum case.
# vaft.omas.compute_eddy_currents(other, [], [])

# 3. Check the model against the measured magnetics before trusting it.
# vaft.omas.plot_magnetics_overview_vacuum(other)
# plt.show()

# 4. Find the breakdown time and the decay index at that moment.
# t_bd = vaft.omas.find_breakdown_onset(other)
# radius, n_index = vaft.omas.compute_decay_index_ods(other, time=t_bd)

# 5. Was the field it formed in stable? Compare with 39915.


## Takeaways and Next Steps

- A discharge is a sequence — breakdown, burn-through, ramp-up — and each phase shows up in
  a different diagnostic. Reading them together is what makes a scenario legible.
- The vessel is part of the circuit. Vessel currents are comparable to the plasma current and
  oppose the coils, and every magnetic measurement sees their sum.
- A vacuum field is a *precondition*, not an afterthought: the loop voltage supplies the
  drive, $B_z$ supplies the balance, and the decay index decides whether that balance holds.
- Check a model against what was measured before deriving anything from it. That is what
  `magnetics_overview_vacuum` is for, and it comes before the physics, not after.
- When a quantity cannot be computed, say so — and when it can, check the estimate you would
  otherwise have trusted. The connection length is traced now, and the scaling it replaced is
  right in shape but scattered by a factor of several point by point, with only half the
  points inside a factor of two — which near the domain edge is the difference between a
  threshold and none.

**Next**: Session 03 takes the discharge past startup and reconstructs the equilibrium it
settled into.
